In [ ]:
%matplotlib widget
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ZTF_FP_Reduction import load_ztf_fp_file, ZTF_FP_Extractor

## 1. Load ZTF Forced Photometry

In [ ]:
FP_FILE = "at2024mfm_ztf_fp.txt"
df = load_ztf_fp_file(FP_FILE, quality_cut=True)
print(f"Loaded {len(df)} epochs  |  filters present: {df['filter'].unique()}")
df.head()

## 2. Compute Magnitudes and Plot Raw Light Curves

In [ ]:
ext = ZTF_FP_Extractor(name="AT2023mfm", df=df)
ext._set_raw_mag_column()
ext._set_raw_magerr_column()

fig, ax = plt.subplots(figsize=(12, 4))
ext.plot_lc(ax, use_corrected=False)
plt.tight_layout()
plt.show()

## 3. Select Baseline Window Interactively

Click and drag on the plot below to select the pre-transient baseline region.
The selected JD range is printed and stored in `ext.baseline_window`.

In [ ]:
ext.get_baseline_window()

In [ ]:
# Confirm the window was captured — should show [jd_start, jd_end].
# If it shows [None, None], drag again or set manually below.
print("Captured baseline window:", ext.baseline_window)

# --- Manual override (uncomment and fill in if the widget didn't fire) ---
# ext._baseline_window = [2458194.5, 2459000.0]

## 4. Apply Baseline Correction and Plot Corrected Light Curve

In [ ]:
assert ext.baseline_window not in (None, [None, None]) and None not in ext.baseline_window, \
    "No window selected — drag on the plot in cell 3 first, then re-run this cell."
print(f"Baseline window: JD {ext.baseline_window[0]:.4f} — {ext.baseline_window[1]:.4f}")
ext.correct_baseline(time_window=ext.baseline_window)

fig, ax = plt.subplots(figsize=(12, 4))
ext.plot_lc(ax, use_corrected=True)
plt.tight_layout()
plt.show()

In [ ]:
out = (
    ext.df[['jd', 'filter', 'baseline_corrected_mag', 'raw_magerr']]
    .dropna(subset=['baseline_corrected_mag'])
    .reset_index(drop=True)
)
out.to_csv("at2023mfm_baseline_corrected_lc.csv", index=False)
print(f"Saved {len(out)} rows to at2023mfm_baseline_corrected_lc.csv")
out.head(10)